# Batch #3 - Model soup: one set of weights, ensemble-class quality

Batch-2 verdict (why this batch exists): the aux head and all five
training-dynamics arms were flat-to-negative in paired tests, so the only
remaining single-model lever is consolidating the seed-ensemble gain in
**weight space**. Model soup = average the fine-tuned parameters of same-recipe
seed runs and evaluate the average as ONE model.

Parts:

1. **A160 seeds 42-49** (8 runs): 42-46 are re-runs (checkpoints from batch-1
   are no longer downloadable; scores double as a reproduction check vs the
   recorded batch-1 values), 47-49 are new. This is the best single-model
   recipe from all batches so far.
2. **B512 seeds 42-44** (3 runs): second regime (512/512), contributes diversity
   to the mixed soup.
3. **Soup suite**: plain soups (A160 pool, B512, mixed) + greedy soup, each
   evaluated as a single model; probs saved for the offline analysis.

**Checklist before Run All** (Settings, right sidebar):

1. **Accelerator: GPU T4 x2**
2. **Internet: On** (git clone + pip + model download)
3. **Add Input -> Datasets -> `amnpham/lgbtqia`**

**How long:** ~3-3.5 h (11 runs + soup evals). Leave the session alone while it
runs; don't press Stop (closing the tab is fine).

**Resume:** each run is skipped when its history CSV already exists; re-running
a batch cell continues where it stopped (same session). A fresh session starts
from scratch.

**Outputs:** per-run histories + prob dumps, soup prob dumps, summaries;
`Try/weights/results_small.zip` refreshed after each phase. When it prints
`BATCH3 COMPLETE`, just tell me - I pull everything via the API.


In [ ]:
import os
import subprocess

REPO_URL = 'https://github.com/AmnO-O/Try.git'
PIN_COMMIT = '37c4c2645aea0dff1e749dc6f762f33d5e843d35'  # aux head, flag-off bit-identical


def sh(cmd):
    print(f'$ {cmd}', flush=True)
    subprocess.run(cmd, shell=True, check=True)


# Re-run safe: after the first run the cwd is already <working>/Try.
if os.path.basename(os.path.abspath('.')) != 'Try':
    if not os.path.isdir('Try'):
        sh(f'git clone {REPO_URL}')
    os.chdir('Try')
if PIN_COMMIT:
    sh('git fetch origin main')
    sh(f'git checkout {PIN_COMMIT}')
sh('git log -1 --format="%H %cd %s" --date=iso')
sh('pip install -q -r requirements.txt')
print('cwd:', os.getcwd(), flush=True)


## 1. Data + GPU check

Finds the StereoQueerEval TSVs under `/kaggle/input` (attach `amnpham/lgbtqia`),
builds the split-42 validation frame and saves labels/langs/videos for the
offline analysis.


In [ ]:
import glob
import sys
import time

hits = sorted(glob.glob('/kaggle/input/**/StereoQueerEval_*_training.tsv', recursive=True))
assert hits, 'StereoQueerEval TSVs not found - attach the dataset (Add Input > amnpham/lgbtqia)'
os.environ['LGBT_DATA_DIR'] = os.path.dirname(hits[0])
os.environ['LGBT_WEIGHTS_DIR'] = os.path.abspath('weights')
os.makedirs('weights', exist_ok=True)
print('data dir:', os.environ['LGBT_DATA_DIR'], flush=True)

import torch
print('torch', torch.__version__, '| cuda', torch.cuda.is_available(), flush=True)
assert torch.cuda.is_available(), 'No GPU - set Settings > Accelerator > GPU T4 x2, then Run All'
print('gpu:', torch.cuda.get_device_name(0), flush=True)

sys.path.insert(0, os.path.abspath('src'))
import numpy as np
import pandas as pd
import data as data_mod
import model_xattn
from model import load_tokenizer
import train_xattn as tx

dev = tx.resolve_device('cuda')
tok = load_tokenizer(model_xattn.DEFAULT_BACKBONE)
df_all = data_mod.load_frame(languages=['EN', 'IT', 'NL'])
_, df_val = data_mod.split_by_video(df_all, 0.1, 42)
labs = df_val['st_y'].astype(int).to_numpy()
assert len(labs) == 738 and int(labs.sum()) == 276, \
    f'unexpected val frame: n={len(labs)} pos={int(labs.sum())}'
np.save('weights/val_labels.npy', labs)
np.save('weights/val_langs.npy', df_val['lang'].to_numpy(dtype=str))
np.save('weights/val_videos.npy', df_val['yt_title'].to_numpy(dtype=str))
print('val:', len(df_val), 'rows | pos', int(labs.sum()),
      '| clusters', df_val['yt_title'].nunique(), flush=True)


## 2. Run lists and helpers

A160 = 160/256 regime (8 seeds), B512 = 512/512 regime (3 seeds). Both are the
same recipe otherwise (dropout 0.3, unfreeze 3, 3 decoder blocks, shared
encoder). `RECORDED` holds the batch-1 val scores for the 42-46 / 42-44 seeds so
each re-run prints its reproduction delta.


In [ ]:
BASE = dict(dropout=0.3, weight_decay=0.01, head_lr=1e-3, xattn_lr=1e-4,
            backbone_lr=2e-5, unfreeze_layers=3, layers=3, shared_encoder=True,
            pooled_v=False, freeze_epochs=0, pool_init_scale=0.02,
            max_len=160, video_max_len=256, no_amp=False)

RUNS_A160 = [dict(BASE, tag=f'A160_s{s}', seed=s, split_seed=42)
             for s in (42, 43, 44, 45, 46, 47, 48, 49)]
RUNS_B512 = [dict(BASE, tag=f'B512_s{s}', seed=s, split_seed=42,
                  max_len=512, video_max_len=512) for s in (42, 43, 44)]
RUNS = RUNS_A160 + RUNS_B512

RECORDED = {'A160_s42': 0.7476, 'A160_s43': 0.7408, 'A160_s44': 0.7295,
            'A160_s45': 0.7443, 'A160_s46': 0.7223,
            'B512_s42': 0.7426, 'B512_s43': 0.7348, 'B512_s44': 0.7390}


def train_one(r):
    hist = f"weights/{r['tag']}_history.csv"
    out = f"weights/{r['tag']}.pt"
    if os.path.exists(hist):
        print(f"-- skip (history exists): {hist}", flush=True)
        return True
    cmd = (
        'python src/train_xattn.py --languages EN IT NL --epochs 15 --patience 5 --batch-size 32 '
        f"--max-len {r['max_len']} --video-max-len {r['video_max_len']} "
        f"--dropout {r['dropout']} --weight-decay {r['weight_decay']} "
        f"--head-lr {r['head_lr']} --xattn-lr {r['xattn_lr']} --backbone-lr {r['backbone_lr']} "
        f"--unfreeze-layers {r['unfreeze_layers']} --layers {r['layers']} "
        f"--pool-init-scale {r['pool_init_scale']} --freeze-epochs {r['freeze_epochs']} "
        f"{'--pooled-v' if r['pooled_v'] else '--no-pooled-v'} "
        f"{'--shared-encoder ' if r['shared_encoder'] else ''}"
        f"{'--no-amp ' if r['no_amp'] else ''}"
        f"--split-seed {r['split_seed']} --seed {r['seed']} "
        f"--out {out} --history {hist}"
    )
    print(f'$ {cmd}', flush=True)
    t0 = time.time()
    rc = subprocess.run(cmd, shell=True, env={**os.environ, 'PYTHONUNBUFFERED': '1'}).returncode
    print(f'== {r["tag"]}: exit {rc} after {(time.time() - t0) / 60:.1f} min', flush=True)
    if rc != 0 or not os.path.exists(hist):
        print(f'!! {r["tag"]} failed', flush=True)
        return False
    h = pd.read_csv(hist)
    bi = h['val_macro_f1'].idxmax()
    best = float(h.loc[bi, 'val_macro_f1'])
    rec = RECORDED.get(r['tag'])
    extra = f' | batch-1 {rec:.4f} (delta {best - rec:+.4f})' if rec else ''
    print(f"   {r['tag']}: best epoch {int(h.loc[bi, 'epoch'])} | "
          f"val_macro_f1 {best:.4f} | acc {h.loc[bi, 'val_acc']:.4f}{extra}", flush=True)
    return True


def dump_probs(r):
    hist = f"weights/{r['tag']}_history.csv"
    out = f"weights/{r['tag']}.pt"
    if not os.path.exists(hist) or not os.path.exists(out):
        print(f'-- no probs for {r["tag"]}', flush=True)
        return
    net = model_xattn.build_model(
        model_xattn.DEFAULT_BACKBONE, unfreeze_layers=r['unfreeze_layers'],
        heads=4, layers=r['layers'], dropout=r['dropout'], ffn_mult=4,
        use_pooled_v=r['pooled_v'], separate_encoders=not r['shared_encoder'],
        keep_frozen_dropout=False, pool_init_scale=r['pool_init_scale'],
    ).to(dev)
    net.load_state_dict(torch.load(out, map_location=dev))
    net.eval()
    loader = tx.make_loader(df_val, tok, 32, r['max_len'], r['video_max_len'])
    o = tx.collect(net, loader, dev, use_amp=True)
    np.save(f"weights/{r['tag']}_val_probs.npy", o['probs'])
    m = tx.metrics_from(o['preds'], o['labels'])
    print(f"   {r['tag']}: reload-eval macro_f1 {m['macro_f1']:.4f} acc {m['acc']:.4f}", flush=True)
    del net
    torch.cuda.empty_cache()


def zip_small():
    import zipfile
    small = [f for f in sorted(os.listdir('weights'))
             if f.endswith(('.csv', '.npy', '.json'))]
    with zipfile.ZipFile('weights/results_small.zip', 'w', zipfile.ZIP_DEFLATED) as zf:
        for f in small:
            zf.write(os.path.join('weights', f), f)
    print(f'wrote weights/results_small.zip ({len(small)} files)', flush=True)


print('planned: A160', len(RUNS_A160), 'runs | B512', len(RUNS_B512), 'runs', flush=True)


## 3. Part 1: A160 seeds 42-49 (~2 h)

Seeds 42-46 re-run for checkpoint access + reproduction check; 47-49 extend
the pool.


In [ ]:
t0 = time.time()
for i, r in enumerate(RUNS_A160, 1):
    print(f'=== [A160 {i}/{len(RUNS_A160)}] {r["tag"]} ===', flush=True)
    try:
        train_one(r)
    except Exception as e:                                    # noqa: BLE001
        print(f'!! train_one({r["tag"]}) raised: {type(e).__name__}: {e}', flush=True)
    try:
        dump_probs(r)
    except Exception as e:                                    # noqa: BLE001
        print(f'!! dump_probs({r["tag"]}) raised: {type(e).__name__}: {e}', flush=True)
print(f'A160 phase finished in {(time.time() - t0) / 3600:.2f} h', flush=True)
zip_small()


## 4. Part 2: B512 seeds 42-44 (~50 min)

Second input regime for the mixed soup.


In [ ]:
t0 = time.time()
for i, r in enumerate(RUNS_B512, 1):
    print(f'=== [B512 {i}/{len(RUNS_B512)}] {r["tag"]} ===', flush=True)
    try:
        train_one(r)
    except Exception as e:                                    # noqa: BLE001
        print(f'!! train_one({r["tag"]}) raised: {type(e).__name__}: {e}', flush=True)
    try:
        dump_probs(r)
    except Exception as e:                                    # noqa: BLE001
        print(f'!! dump_probs({r["tag"]}) raised: {type(e).__name__}: {e}', flush=True)
print(f'B512 phase finished in {(time.time() - t0) / 3600:.2f} h', flush=True)
zip_small()


## 5. Part 3: soup suite (~15-25 min)

Average the fine-tuned weights of same-recipe seeds and evaluate the average as
a **single model**:

- `soup_A160` - all available A160 seeds, evaluated at 160/256
- `soup_B512` - the B512 seeds, evaluated at 512/512
- `soup_mix` - A160 + B512 pool, evaluated at both regimes
- `soup_greedy` - fixed-order greedy add (accept a seed only if val improves)

Prob dumps of every soup are saved for the offline analysis (they can also be
prob-averaged with the seed pool later).


In [ ]:
import gc


def soup_state_dicts(paths):
    out = None
    n = 0
    for p in paths:
        sd = torch.load(p, map_location='cpu')
        if out is None:
            out = {k: (v.float().clone() if v.is_floating_point() else v.clone())
                   for k, v in sd.items()}
        else:
            for k in out:
                v = sd.get(k)
                if v is not None and out[k].is_floating_point():
                    out[k].add_(v.float())
        n += 1
        del sd
        gc.collect()
    for k in out:
        if out[k].is_floating_point():
            out[k].div_(n)
    return out


def eval_sd(sd, max_len, vmax, name, save=True):
    net = model_xattn.build_model(
        model_xattn.DEFAULT_BACKBONE, unfreeze_layers=0,
        heads=4, layers=3, dropout=0.3, ffn_mult=4,
        use_pooled_v=False, separate_encoders=False,
        keep_frozen_dropout=False, pool_init_scale=0.02,
    )
    missing, unexpected = net.load_state_dict(sd, strict=False)
    if missing or unexpected:
        print(f'   note: missing={missing[:4]} unexpected={unexpected[:4]}', flush=True)
    net.to(dev).eval()
    loader = tx.make_loader(df_val, tok, 32, max_len, vmax)
    o = tx.collect(net, loader, dev, use_amp=True)
    m = tx.metrics_from(o['preds'], o['labels'])
    if save:
        np.save(f'weights/{name}_val_probs.npy', o['probs'])
    print(f"== {name}: macro_f1 {m['macro_f1']:.4f} | acc {m['acc']:.4f}", flush=True)
    del net
    torch.cuda.empty_cache()
    return float(m['macro_f1'])


a160 = [f'A160_s{s}' for s in range(42, 50) if os.path.exists(f'weights/A160_s{s}.pt')]
b512 = [f'B512_s{s}' for s in (42, 43, 44) if os.path.exists(f'weights/B512_s{s}.pt')]
print('soup pool A160:', a160, '| B512:', b512, flush=True)

if a160:
    eval_sd(soup_state_dicts([f'weights/{t}.pt' for t in a160]), 160, 256, 'soup_A160')
else:
    print('!! no A160 checkpoints - skipping soup_A160', flush=True)
if b512:
    eval_sd(soup_state_dicts([f'weights/{t}.pt' for t in b512]), 512, 512, 'soup_B512')
    mix = soup_state_dicts([f'weights/{t}.pt' for t in a160 + b512])
    eval_sd(mix, 160, 256, 'soup_mix_len160')
    eval_sd(mix, 512, 512, 'soup_mix_len512')
    del mix
    torch.cuda.empty_cache()

current = []
best = -1.0
for t in a160:
    trial = current + [t]
    f1 = eval_sd(soup_state_dicts([f'weights/{c}.pt' for c in trial]), 160, 256,
                 f'soup_greedy_try_{len(trial)}', save=False)
    if f1 > best:
        best, current = f1, trial
        print(f'   accepted {t} -> {current} ({f1:.4f})', flush=True)
    else:
        print(f'   rejected {t} ({f1:.4f} <= {best:.4f})', flush=True)
if current:
    eval_sd(soup_state_dicts([f'weights/{c}.pt' for c in current]), 160, 256,
            'soup_greedy_final')
print('greedy final:', current, 'macro_f1', round(best, 4), flush=True)


def prob_ens(tags):
    ps = [np.load(f'weights/{t}_val_probs.npy') for t in tags
          if os.path.exists(f'weights/{t}_val_probs.npy')]
    if not ps:
        return None
    p = np.mean(ps, axis=0)
    return tx.metrics_from(p.argmax(1), labs)['macro_f1']


pe = prob_ens(a160)
print('reference prob-ensemble A160 pool:',
      round(pe, 4) if pe is not None else 'n/a', flush=True)
if b512:
    print('reference prob-ensemble A160+B512 pool:',
          round(prob_ens(a160 + b512), 4), flush=True)


## 6. Summary


In [ ]:
try:
    rows = []
    for r in RUNS:
        hist = f"weights/{r['tag']}_history.csv"
        if os.path.exists(hist):
            h = pd.read_csv(hist)
            bi = h['val_macro_f1'].idxmax()
            rows.append(dict(tag=r['tag'], seed=r['seed'], epochs=len(h),
                             best_epoch=int(h.loc[bi, 'epoch']),
                             best_f1=round(float(h.loc[bi, 'val_macro_f1']), 4),
                             best_acc=round(float(h.loc[bi, 'val_acc']), 4)))
        else:
            rows.append(dict(tag=r['tag'], seed=r['seed'], epochs=0,
                             best_epoch=None, best_f1=None, best_acc=None))
    summary = pd.DataFrame(rows)
    print(summary.to_string(index=False), flush=True)
    summary.to_csv('weights/run_summary.csv', index=False)

    soup_rows = []
    for f in sorted(os.listdir('weights')):
        if f.startswith('soup_') and f.endswith('_val_probs.npy'):
            name = f[:-len('_val_probs.npy')]
            p = np.load(f'weights/{f}')
            m = tx.metrics_from(p.argmax(1), labs)
            soup_rows.append(dict(soup=name, macro_f1=round(float(m['macro_f1']), 4),
                                  acc=round(float(m['acc']), 4)))
            print(f"SOUP {name}: macro_f1 {m['macro_f1']:.4f} | acc {m['acc']:.4f}", flush=True)
    pd.DataFrame(soup_rows).to_csv('weights/soup_summary.csv', index=False)
except Exception as e:                                        # noqa: BLE001
    print(f'!! summary failed: {type(e).__name__}: {e}', flush=True)

print('files in weights/:', sorted(os.listdir('weights')), flush=True)
zip_small()
print('BATCH3 COMPLETE - tell the assistant; outputs can be pulled via the API', flush=True)
